<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDCNNImageClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [165]:
import numpy as np
import numpy.lib.stride_tricks as lst
import matplotlib.pyplot as plt
import math
import random
import re
import cv2
import csv
from google.colab.patches import cv2_imshow
%matplotlib inline

In [166]:
# Hyperparameters
START_SIZE = 256
BATCH_SIZE = 6
WIN_SIZE_L1 = (5, 5)
NUM_FEATURES_L1 = 128
MAX_POOLING_SHRINK_L1 = 4
WIN_SIZE_L2 = (5, 5)
NUM_FEATURES_L2 = 64
MAX_POOLING_SHRINK_L2 = 4
WIN_SIZE_L3 = (5, 5)
NUM_FEATURES_L3 = 8
MAX_POOLING_SHRINK_L3 = 2

In [167]:
# # Opening the File
# dataset = []
# with open('/content/slidemdimageds.csv', mode='r', newline='') as file:
#     reader = csv.reader(file)
#     for row in reader:
#         if len(row) == 3:
#           dataset.append(row)

In [168]:
# # Seperating Inputs and Outputs
# last_num_int = lambda a: int(a[-1])
# inputs_seperation = lambda a: a[:2]
# Y = list(map(last_num_int, dataset))
# inputs = list(map(inputs_seperation, dataset))

In [169]:
# # Preprocessing Text
# stoi = {chr(i): (i - ord('a') + 1) for i in range(ord('a'), ord('z') + 1)}
# stoi['.'] = 0
# itos = {ix: letter for letter, ix in stoi.items()}

In [170]:
# # Converting All Words Into Numbers
# def tok(word):
#   word = re.sub('[^a-z]', '', word.lower())
#   letters = list(word)
#   enc_letters = list(map(lambda a: stoi[a], letters))
#   if len(enc_letters) < MAX_WORD_LEN:
#     enc_letters.extend([0] * (MAX_WORD_LEN - len(enc_letters)))
#   return enc_letters

In [171]:
# # Replacing All Text with Tokenized Words
# inputs = [[input[0], tok(input[1])] for input in inputs]

In [172]:
# Preprocessing Image
images = []
for i in range(0,6):
    img_path = f"image_{i}.jpeg"
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        resized_img = cv2.resize(img, (START_SIZE, START_SIZE))
        images.append(resized_img)

X = np.array(images).astype(np.float32)
X /= np.max(X)
Y = np.array([1,0,1,1,1,0])
Y = Y.reshape(-1, 1)

In [173]:
# PARAPARAMETER!!!
W1 = np.random.randn(*WIN_SIZE_L1, NUM_FEATURES_L1) * 0.01
g1 = np.ones(NUM_FEATURES_L1)
b1 = np.zeros_like(g1)
W2 = np.random.randn(*WIN_SIZE_L2, NUM_FEATURES_L1, NUM_FEATURES_L2) * 0.01
g2 = np.ones(NUM_FEATURES_L2)
b2 = np.zeros_like(g2)
W3 = np.random.randn(*WIN_SIZE_L3, NUM_FEATURES_L2, NUM_FEATURES_L3) * 0.01
g3 = np.ones(NUM_FEATURES_L3)
b3 = np.zeros_like(g3)
W4 = np.random.randn(NUM_FEATURES_L3 * (START_SIZE // MAX_POOLING_SHRINK_L1 // MAX_POOLING_SHRINK_L2 // MAX_POOLING_SHRINK_L3) ** 2, 1) * 0.01

In [174]:
### Forward Pass

## Layer 1
# Padding + Conv
pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
img_padded1 = np.pad(X, ((0,0), pad1, pad1), "constant", constant_values=0)
win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])
preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
preln1 = preln1.reshape(BATCH_SIZE, X.shape[1], X.shape[1], NUM_FEATURES_L1)
# Layer Norm
lnmean1 = np.mean(preln1, axis = -1, keepdims = True)
lnvar1 = np.var(preln1, axis = -1, keepdims = True)
lnraw1 = (preln1 - lnmean1)/np.sqrt(lnvar1 + 1e-5)
ln1 = g1 * lnraw1 + b1
# ReLu
rl1 = np.maximum(np.zeros_like(ln1), ln1)
# Max Pooling
rl1 = rl1.reshape(BATCH_SIZE, NUM_FEATURES_L1, rl1.shape[1], -1)
img_pieced1 = rl1.reshape(BATCH_SIZE, NUM_FEATURES_L1, rl1.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, rl1.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
pooled_img1 = np.max(img_pieced1, axis=(-1, -3))

## Layer 2
# Padding + Conv
pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)
win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
win_l2r = np.swapaxes(np.expand_dims(np.ascontiguousarray(win_l2), -1), 1, -1)
win_l2r = win_l2r.squeeze(1)
win_l2r = win_l2r.reshape(-1, win_l2r.shape[-2] * win_l2r.shape[-1] * win_l2r.shape[-3])
preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
preln2 = preln2.reshape(BATCH_SIZE, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)
# Layer Norm
lnmean2 = np.mean(preln2, axis = -1, keepdims = True)
lnvar2 = np.var(preln2, axis = -1, keepdims = True)
lnraw2 = (preln2 - lnmean2)/np.sqrt(lnvar2 + 1e-5)
ln2 = g2 * lnraw2 + b2
# ReLu
rl2 = np.maximum(np.zeros_like(ln2), ln2)
# Max Pooling
rl2 = rl2.reshape(BATCH_SIZE, NUM_FEATURES_L2, rl2.shape[1], -1)
img_pieced2 = rl2.reshape(BATCH_SIZE, NUM_FEATURES_L2, rl2.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2, rl2.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2)
pooled_img2 = np.max(img_pieced2, axis=(-1, -3))

## Layer 3
# Padding + Conv
pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)
win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
win_l3r = np.swapaxes(np.expand_dims(np.ascontiguousarray(win_l3), -1), 1, -1)
win_l3r = win_l3r.squeeze(axis=1)
win_l3r = win_l3r.reshape(-1, win_l3r.shape[-2] * win_l3r.shape[-1] * win_l3r.shape[-3])
preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
preln3 = preln3.reshape(BATCH_SIZE, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)
# Layer Norm
lnmean3 = np.mean(preln3, axis = -1, keepdims = True)
lnvar3 = np.var(preln3, axis = -1, keepdims = True)
lnraw3 = (preln3 - lnmean3)/np.sqrt(lnvar3 + 1e-5)
ln3 = g3 * lnraw3 + b3
# ReLu
rl3 = np.maximum(np.zeros_like(ln3), ln3)
# Max Pooling
rl3 = rl3.reshape(BATCH_SIZE, NUM_FEATURES_L3, rl3.shape[1], -1)
img_pieced3 = rl3.reshape(BATCH_SIZE, NUM_FEATURES_L3, rl3.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3, rl3.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3)
pooled_img3 = np.max(img_pieced3, axis=(-1, -3))

## Layer 4
inputs = pooled_img3.reshape(BATCH_SIZE, -1)
raw_pred = inputs @ W4

## Sigmoid
probs = 1/(1 + (np.e ** -(raw_pred)))

## Loss
loss = -(Y * np.log(probs + 1e-5) + (1 - Y) * np.log(1 - probs + 1e-5))
loss.mean()

(6, 1)

In [176]:
# Me Self Inventing Max Pooling
# size = random.randint(1, 20)
# size = size if size % 2 == 0 else size + 1
size = 8

# img = np.random.randn(size**2).reshape(size,size)
img = np.arange(size**2).reshape(size, size)
img = img.astype(np.float32)
wins = lst.sliding_window_view(img, window_shape=(2,2), axis = (0,1))
img_pieced = (wins[0::2, 0::2, :, :]).reshape(-1, 2 * 2)
max_pooling = np.max(img_pieced, axis = 1)
max_pooling = max_pooling.reshape(-1, int(np.sqrt(max_pooling.shape[0])))

## Try #2 with Reshaping
imr = img.reshape(size // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, size // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
img_pooled = np.max(imr, axis = (1, -1))
# Apparently Possible with Reshaping
### This fails. It only works w 4x4 & 8x8
# MAX_POOLING_SHRINK_L1 = 2
# win_area1 = MAX_POOLING_SHRINK_L1 ** 2
# img_r = np.ascontiguousarray(img).reshape(-1, MAX_POOLING_SHRINK_L1)
# new_void = np.dtype((np.void, img.dtype.itemsize * MAX_POOLING_SHRINK_L1))
# img_r = img_r.view(new_void)
# img_r = img_r.reshape(size, -1)
# img_rt = np.transpose(img_r)
# img_rt = np.ascontiguousarray(img_rt).view(np.float32)
# img_prt = img_rt.reshape(-1, win_area1)
# bigger_void = np.dtype((np.void, img.dtype.itemsize * win_area1))
# img_prt = np.ascontiguousarray(img_prt).view(bigger_void)
# sort = np.arange(size ** 2 / win_area1) % win_area1 * win_area1 + np.arange(size ** 2 / win_area1) // win_area1
# original_sort = np.argsort(sort)
# img_piecedr = img_prt[original_sort].view(np.float32)
# max_poolingr = np.max(img_piecedr, axis = 1)
# max_poolingr = max_poolingr.reshape(-1, size // MAX_POOLING_SHRINK_L1)
img, img_pooled

(array([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.],
        [32., 33., 34., 35., 36., 37., 38., 39.],
        [40., 41., 42., 43., 44., 45., 46., 47.],
        [48., 49., 50., 51., 52., 53., 54., 55.],
        [56., 57., 58., 59., 60., 61., 62., 63.]], dtype=float32),
 array([[27., 31.],
        [59., 63.]], dtype=float32))